# PROCESS large tokamak — tungsten sensitivity

The quantitative form of PROCESS's `single_model_evaluation.ex.py`, which raises
tungsten and observes radiated power rise and separatrix power fall.
Five evaluation-mode runs in `reference/tungsten_sweep/`.

A *response* is a stronger test than an absolute value: an offset shared by every
point cancels out of the trend.

In [ ]:
# Validated categorical order (dataviz palette): slot 1 blue, slot 2 orange.
# Two series only, assigned in fixed order and never cycled.
FUSDB, PROCESS = '#2a78d6', '#eb6834'
import matplotlib.pyplot as plt, numpy as np, sys
sys.path.insert(0, '.')
plt.rcParams.update({'figure.dpi':120, 'axes.spines.top':False, 'axes.spines.right':False,
                     'axes.grid':True, 'grid.alpha':0.25, 'grid.linewidth':0.6,
                     'axes.labelcolor':'#52514e', 'text.color':'#0b0b0b',
                     'xtick.color':'#52514e', 'ytick.color':'#52514e', 'font.size':9})

from _process_compare import compare
from _process_fixture import tungsten_sweep
pts = tungsten_sweep()
res = {tag: compare(path) for tag, path in pts.items()}
W = np.array([float(t.split('_')[1]) for t in res])
order = np.argsort(W); tags = np.array(list(res))[order]; W = W[order]
sustained = np.array([res[t][1]['regime'] == 'h_mode' for t in tags])
for t in tags: print(f"{t:12s} regime={res[t][1]['regime']}")

## Radiated and separatrix power vs tungsten

Separate axes per quantity — never a dual-axis chart. Restricted to the four
H-mode-sustained points; the fifth sits on a different confinement branch and
is the subject of the panel below.

In [ ]:
# Only the H-mode-sustained points: once a point drops out of H-mode the
# confinement scaling changes underneath it, so its powers are not on this
# curve at all (fusdb's demoted solve puts P_sep at ~850 MW).  Plotting it
# here would squash the real data against the axis; where the branch ends is
# the subject of the next panel instead.
Ws, ts = W[sustained], tags[sustained]
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.6), sharex=True)
for ax, key, lab in zip(axes, ['P_rad','P_sep'],
                        ['radiated power  [MW]','separatrix power  [MW]']):
    for side, color, name in (('process', PROCESS, 'PROCESS'), ('fusdb', FUSDB, 'fusdb')):
        y = np.array([res[t][0][key][side] for t in ts])/1e6
        ax.plot(Ws, y, color=color, lw=2, marker='o', ms=6, label=name)
    ax.set_xscale('log'); ax.set_xlabel('tungsten fraction  $n_W/n_e$'); ax.set_ylabel(lab)
    # Tick only at the sampled values -- log minor ticks collide at this range.
    ax.set_xticks(Ws); ax.set_xticklabels([f'{w:.0e}' for w in Ws], fontsize=8)
    ax.set_xticks([], minor=True)
    ax.legend(frameon=False, fontsize=8)
axes[0].set_title('Radiation rises and separatrix power falls, in both codes',
                  loc='left', pad=10)
fig.tight_layout()

## Where H-mode stops being sustainable

The two codes agree on this, which is the point of the panel below. PROCESS in
evaluation mode keeps using whichever confinement scaling its input declares; fusdb
*derives* the regime and drops out of H-mode. But PROCESS's own reported `P_sep`
falls below its own `P_LH` at exactly the same point, so the two agree on the
physics even though only one of them acts on it.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.8))
ps = np.array([res[t][0]['P_sep']['process'] for t in tags])/1e6
lh = np.array([res[t][0]['P_LH']['process'] for t in tags])/1e6
below = ps < lh
# Shade the region PROCESS's own numbers declare un-sustainable.
if below.any():
    ax.axvspan(W[below][0]*0.72, W[-1]*1.35, color='#e34948', alpha=0.09, lw=0)
ax.plot(W, ps, color=PROCESS, lw=2, marker='o', ms=6, label='PROCESS $P_{sep}$')
ax.plot(W, lh, color='#52514e', lw=1.6, ls='--', marker='', label='PROCESS $P_{LH}$ threshold')
# fusdb's verdict is a regime, not a power: mark where it keeps H-mode.
keep = np.array([res[t][1]['regime'] == 'h_mode' for t in tags])
ax.scatter(W[keep], ps[keep], s=140, facecolors='none', edgecolors=FUSDB, lw=1.8,
           zorder=4, label='fusdb keeps H-mode')
ax.set_xscale('log'); ax.set_xlabel('tungsten fraction  $n_W/n_e$')
ax.set_xticks(W); ax.set_xticklabels([f'{w:.0e}' for w in W], fontsize=8)
ax.set_xticks([], minor=True)
ax.set_ylabel('power  [MW]'); ax.set_ylim(0, max(ps.max(), lh.max())*1.18)
ax.set_xlim(W[0]*0.8, W[-1]*1.25)
if below.any():
    ax.text(W[below][0]*0.78, max(ps.max(), lh.max())*1.10,
            'PROCESS: $P_{sep}<P_{LH}$', fontsize=8, color='#e34948')
ax.legend(frameon=False, fontsize=8, loc='lower left')
ax.set_title('Both codes put the L-H crossing between 5e-5 and 1e-4', loc='left', pad=10)
fig.tight_layout()